In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

BRONZE_TABLE = "weather.bronze.metar"
DAILY_PATH = "/Volumes/weather/raw/noaa_volume/json/daily/"

# Leer schema oficial de bronze
bronze_schema = spark.table(BRONZE_TABLE).schema

# Leer daily con schema forzado
daily_df = (
    spark.read
         .schema(bronze_schema)
         .option("multiLine", True)
         .json(DAILY_PATH)
)

# Opcional pero recomendable: dedupe interno del batch
daily_df = daily_df.dropDuplicates(["icaoId", "obsTime"])

delta_table = DeltaTable.forName(spark, BRONZE_TABLE)

(
    delta_table.alias("t")
    .merge(
        daily_df.alias("s"),
        "t.icaoId = s.icaoId AND t.obsTime = s.obsTime"
    )
    .whenNotMatchedInsertAll()
    .execute()
)